# Machine Learning - Assignment 2

## Classification Models and Streamlit Deployment

**Programme:** M.Tech in Data Science

**Course:** Machine Learning (S2-25_DSECLZG565)  
**Assignment:** Assignment 2

### Objective

The objective of this assignment is to implement multiple classification machine learning models on a public classification dataset, evaluate their performance using multiple evaluation metrics, compare the models, and deploy the trained models using a Streamlit web application.

### Models Implemented

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbor Classifier
4. Gaussian Naive Bayes Classifier
5. Random Forest Classifier

### Evaluation Metrics

- Accuracy
- AUC Score
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

# 1. Problem Statement

Classification is a supervised machine learning task in which a machine learning model learns from labelled data and predicts the class or category of new observations.

In this assignment, a public classification dataset is selected and multiple classification algorithms are implemented on the same dataset. The models are evaluated using Accuracy, AUC Score, Precision, Recall, F1 Score, and Matthews Correlation Coefficient (MCC).

The objective is to compare the performance of different classification algorithms and identify the best-performing model for the selected dataset.

##Import Required Libraries

The required Python libraries are imported for data manipulation, visualization, preprocessing, model building, evaluation, and model persistence.

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

print("All libraries imported successfully.")

All libraries imported successfully.


# 2. Dataset Selection and Description

The Breast Cancer Wisconsin (Diagnostic) dataset from the UCI Machine Learning Repository is used for this assignment.

The dataset contains 569 instances and 30 predictive numerical features. The target variable is `diagnosis`, which represents whether the diagnosis is benign or malignant.

The dataset satisfies the minimum requirements of at least 500 instances and at least 12 features.

In [24]:
# Save the downloaded UCI dataset locally

df.to_csv("breast_cancer_wisconsin.csv", index=False)

print("Dataset saved locally as breast_cancer_wisconsin.csv")
print("Dataset shape:", df.shape)

Dataset saved locally as breast_cancer_wisconsin.csv
Dataset shape: (569, 32)


# 3. Dataset Verification

In [27]:
print("Number of instances:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Number of predictive features:", df.shape[1] - 2)

print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["diagnosis"].value_counts())

print("\nDataset preview:")
display(df.head())

Number of instances: 569
Number of columns: 32
Number of predictive features: 30

Missing values: 0
Duplicate rows: 0

Target distribution:
diagnosis
B    357
M    212
Name: count, dtype: int64

Dataset preview:


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


# 4. Data Preprocessing

The `id` column is removed because it is an identification field and does not provide meaningful predictive information.

The target variable `diagnosis` is encoded as:
- B (Benign) = 0
- M (Malignant) = 1

In [30]:
# Remove ID column
df_model = df.drop(columns=["id"]).copy()

# Encode target
df_model["diagnosis"] = df_model["diagnosis"].map({
    "B": 0,
    "M": 1
})

# Separate features and target
X = df_model.drop(columns=["diagnosis"])
y = df_model["diagnosis"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

Feature matrix shape: (569, 30)
Target vector shape: (569,)


# 5. Train-Test Split

The dataset is divided into training and testing sets using an 80:20 split.

Stratification is used to preserve the proportion of the two target classes. A fixed random state is used for reproducibility.

In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 455
Testing samples: 114


# 6. Feature Scaling

StandardScaler is used for models that are sensitive to feature magnitude, particularly Logistic Regression and K-Nearest Neighbors.

The scaler is fitted only on the training data and then applied to both training and testing data.

In [38]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed successfully.")

Feature scaling completed successfully.


# 7. Classification Model Implementation

Five classification models are implemented using the same training and testing dataset:

1. Logistic Regression
2. Decision Tree Classifier
3. K-Nearest Neighbor Classifier
4. Gaussian Naive Bayes Classifier
5. Random Forest Classifier

In [41]:
# Initialize models

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        random_state=42
    ),
    
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    
    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),
    
    "Naive Bayes": GaussianNB(),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )
}

# Train models
models["Logistic Regression"].fit(X_train_scaled, y_train)
models["KNN"].fit(X_train_scaled, y_train)

models["Decision Tree"].fit(X_train, y_train)
models["Naive Bayes"].fit(X_train, y_train)
models["Random Forest"].fit(X_train, y_train)

print("All 5 models trained successfully.")

All 5 models trained successfully.


# 8. Model Evaluation

The performance of each classification model is evaluated using the following metrics:

- Accuracy
- AUC Score
- Precision
- Recall
- F1 Score
- Matthews Correlation Coefficient (MCC)

In [44]:
results = []

# Select the appropriate test data for each model
test_data_for_model = {
    "Logistic Regression": X_test_scaled,
    "Decision Tree": X_test,
    "KNN": X_test_scaled,
    "Naive Bayes": X_test,
    "Random Forest": X_test
}

for model_name, model in models.items():
    
    X_eval = test_data_for_model[model_name]
    
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results.append({
        "ML Model Name": model_name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })

results_df = pd.DataFrame(results)

print("Model evaluation completed.")
display(results_df.round(4))

Model evaluation completed.


,ML Model Name,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
1,Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492
2,KNN,0.9561,0.9823,0.9744,0.9048,0.9383,0.9058
3,Naive Bayes,0.9386,0.9934,1.0000,0.8333,0.9091,0.8715
4,Random Forest,0.9649,0.9942,1.0000,0.9048,0.9500,0.9258


# 9. Model Performance Comparison

The following table compares all implemented classification models using the required evaluation metrics.

In [47]:
display(
    results_df.style.format({
        "Accuracy": "{:.4f}",
        "AUC": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}",
        "MCC": "{:.4f}"
    })
)

,ML Model Name,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.9649,0.9960,0.9750,0.9286,0.9512,0.9245
1,Decision Tree,0.9298,0.9246,0.9048,0.9048,0.9048,0.8492
2,KNN,0.9561,0.9823,0.9744,0.9048,0.9383,0.9058
3,Naive Bayes,0.9386,0.9934,1.0000,0.8333,0.9091,0.8715
4,Random Forest,0.9649,0.9942,1.0000,0.9048,0.9500,0.9258


# 10. Overall Winner

The overall best-performing model is selected based on its overall performance across Accuracy, AUC, Precision, Recall, F1 Score, and MCC.

In [50]:
# Calculate average performance across all evaluation metrics

metric_columns = [
    "Accuracy",
    "AUC",
    "Precision",
    "Recall",
    "F1 Score",
    "MCC"
]

results_df["Average Score"] = results_df[metric_columns].mean(axis=1)

winner = results_df.loc[
    results_df["Average Score"].idxmax(),
    "ML Model Name"
]

print("Overall Winner:", winner)

print("\nAverage scores:")
display(
    results_df[["ML Model Name", "Average Score"]]
    .sort_values("Average Score", ascending=False)
    .round(4)
)

Overall Winner: Logistic Regression

Average scores:


,ML Model Name,Average Score
0,Logistic Regression,0.9567
4,Random Forest,0.9566
2,KNN,0.9436
3,Naive Bayes,0.9243
1,Decision Tree,0.9030


# 11. Confusion Matrix and Classification Report

The confusion matrix and classification report are generated for the overall best-performing model.

In [54]:
# Get predictions from the winning model

winner_model = models[winner]
winner_X_test = test_data_for_model[winner]

winner_predictions = winner_model.predict(winner_X_test)

print("Classification Report:")
print(classification_report(
    y_test,
    winner_predictions,
    target_names=["Benign", "Malignant"]
))

cm = confusion_matrix(y_test, winner_predictions)

print("\nConfusion Matrix:")
print(cm)

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        72
   Malignant       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


Confusion Matrix:
[[71  1]
 [ 3 39]]


# 12. Observations on Model Performance

The observations below are based on the evaluation metrics obtained from the test dataset.

In [57]:
for _, row in results_df.sort_values(
    "Average Score", ascending=False
).iterrows():
    
    print(
        f"{row['ML Model Name']}: "
        f"Accuracy={row['Accuracy']:.4f}, "
        f"AUC={row['AUC']:.4f}, "
        f"F1={row['F1 Score']:.4f}, "
        f"MCC={row['MCC']:.4f}"
    )

print("\nOverall Winner:", winner)

Logistic Regression: Accuracy=0.9649, AUC=0.9960, F1=0.9512, MCC=0.9245
Random Forest: Accuracy=0.9649, AUC=0.9942, F1=0.9500, MCC=0.9258
KNN: Accuracy=0.9561, AUC=0.9823, F1=0.9383, MCC=0.9058
Naive Bayes: Accuracy=0.9386, AUC=0.9934, F1=0.9091, MCC=0.8715
Decision Tree: Accuracy=0.9298, AUC=0.9246, F1=0.9048, MCC=0.8492

Overall Winner: Logistic Regression


# 13. Saving Trained Models

The trained models and feature scaler are saved so that they can be loaded later by the Streamlit application without retraining the models.

In [60]:
# Create model directory
os.makedirs("model", exist_ok=True)

# Save models
for model_name, model in models.items():
    
    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        + ".pkl"
    )
    
    # Save models in a standardized way
    joblib.dump(model, os.path.join("model", filename))

# Save scaler
joblib.dump(scaler, "model/scaler.pkl")

print("All models and scaler saved successfully.")

print("\nSaved files:")
for file in os.listdir("model"):
    print(file)

All models and scaler saved successfully.

Saved files:
logistic_regression.pkl
decision_tree.pkl
knn.pkl
naive_bayes.pkl
random_forest.pkl
scaler.pkl


# 14. Export Test Dataset

The test dataset is exported as a CSV file for use in the Streamlit application.

In [63]:
test_data = X_test.copy()
test_data["diagnosis"] = y_test.values

test_data.to_csv("test_data.csv", index=False)

print("test_data.csv created successfully.")
print("Test data shape:", test_data.shape)

test_data.csv created successfully.
Test data shape: (114, 31)


# 15. Conclusion

In this assignment, five classification machine learning models were implemented on the selected UCI Breast Cancer Wisconsin (Diagnostic) dataset.

The models were evaluated using Accuracy, AUC, Precision, Recall, F1 Score, and Matthews Correlation Coefficient (MCC).

The comparative analysis was used to identify the overall best-performing model on the test dataset.

The trained models, scaler, and test dataset were saved for integration with the Streamlit web application.

# 16. Saving Models and Test Data

The trained classification models, feature scaler, and test dataset are saved for integration with the Streamlit web application.

In [67]:
# Create model directory
os.makedirs("model", exist_ok=True)

# Save trained models
joblib.dump(models["Logistic Regression"], "model/logistic_regression.pkl")
joblib.dump(models["Decision Tree"], "model/decision_tree.pkl")
joblib.dump(models["KNN"], "model/knn.pkl")
joblib.dump(models["Naive Bayes"], "model/naive_bayes.pkl")
joblib.dump(models["Random Forest"], "model/random_forest.pkl")

# Save scaler
joblib.dump(scaler, "model/scaler.pkl")

# Create test data with target column
test_data = X_test.copy()
test_data["diagnosis"] = y_test.values

test_data.to_csv("test_data.csv", index=False)

print("Models saved successfully.")
print("Scaler saved successfully.")
print("Test data saved successfully.")
print("Test data shape:", test_data.shape)

Models saved successfully.
Scaler saved successfully.
Test data saved successfully.
Test data shape: (114, 31)
